In [1]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations
import sys
sys.path.append("..")

# ── Load data ────────────────────────────────────────────────────────
with open("../data/puzzles_labeled.json", "r") as f:
    puzzles = json.load(f)

# fix words and remove emoji puzzle
for puzzle in puzzles:
    words_from_groups = []
    for group in puzzle["groups"]:
        words_from_groups.extend(group["members"])
    puzzle["words"] = words_from_groups
puzzles = [p for p in puzzles if p["puzzle_id"] != 295]

# ── Train/test split by date ─────────────────────────────────────────
puzzles_sorted = sorted(puzzles, key=lambda p: p["date"])
split_idx = int(len(puzzles_sorted) * 0.80)
train_puzzles = puzzles_sorted[:split_idx]
test_puzzles = puzzles_sorted[split_idx:]

print(f"Train: {len(train_puzzles)} puzzles")
print(f"Test:  {len(test_puzzles)} puzzles")
print(f"Train date range: {train_puzzles[0]['date']} to {train_puzzles[-1]['date']}")
print(f"Test date range:  {test_puzzles[0]['date']} to {test_puzzles[-1]['date']}")

Train: 180 puzzles
Test:  46 puzzles
Train date range: 2023-10-16 to 2024-04-18
Test date range:  2024-04-19 to 2024-06-03


In [2]:
# ── Load MPNet and compute embeddings ────────────────────────────────
print("Loading MPNet...")
model = SentenceTransformer("all-mpnet-base-v2")
print("Model loaded")

print("Computing embeddings...")
for puzzle in puzzles_sorted:
    embeddings = model.encode(puzzle["words"], show_progress_bar=False)
    puzzle["embeddings"] = embeddings

print(f"Embeddings computed for {len(puzzles_sorted)} puzzles")

Loading MPNet...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded
Computing embeddings...
Embeddings computed for 226 puzzles


In [3]:
from solver.candidates import generate_candidates
from solver.scoring import score_all_candidates
from solver.search import greedy_solve, beam_solve, get_tau_estimate
from solver.evaluate import run_evaluation

print("Solver modules imported successfully")

Solver modules imported successfully


[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>
[nltk_data] Error loading omw-1.4: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>


In [4]:
# ── Full ablation evaluation on test set ─────────────────────────────
import copy
from solver.candidates import generate_candidates
from solver.scoring import score_all_candidates
from solver.search import greedy_solve, beam_solve, get_tau_estimate
from solver.lexical import add_lexical_scores
from solver.evaluate import run_evaluation, evaluate_prediction
from solver.feedback import apply_feedback, simulate_feedback

# best weights from grid search
ALPHA = 1.0
BETA  = 0.0
GAMMA = 0.05
ETA   = 0.3

# precompute candidates for test puzzles once
print("Precomputing test candidates...")
test_precomputed = []
for puzzle in test_puzzles:
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    candidates = generate_candidates(words, embeddings)
    tau = get_tau_estimate(candidates)
    test_precomputed.append({
        "puzzle": puzzle,
        "candidates": candidates,
        "tau": tau
    })
print(f"Done — {len(test_precomputed)} puzzles precomputed")

Precomputing test candidates...
Done — 46 puzzles precomputed


In [5]:
# ── Solver 1: Greedy ─────────────────────────────────────────────────
solved_s1, correct_s1, top1_s1 = [], [], []
for item in test_precomputed:
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    predicted = greedy_solve(words, embeddings, candidates)
    result = evaluate_prediction(predicted, puzzle["groups"])
    solved_s1.append(result["solved"])
    correct_s1.append(result["n_correct"])
    top1_s1.append(result["top1_correct"])

# ── Solver 2: Beam only ──────────────────────────────────────────────
solved_s2, correct_s2, top1_s2 = [], [], []
for item in test_precomputed:
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    predicted = beam_solve(words, embeddings, candidates, beam_width=25)
    result = evaluate_prediction(predicted, puzzle["groups"])
    solved_s2.append(result["solved"])
    correct_s2.append(result["n_correct"])
    top1_s2.append(result["top1_correct"])

# ── Solver 3: Beam + penalty ─────────────────────────────────────────
solved_s3, correct_s3, top1_s3 = [], [], []
for item in test_precomputed:
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    tau = item["tau"]
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    candidates = score_all_candidates(candidates, tau, ALPHA, BETA, GAMMA)
    predicted = beam_solve(words, embeddings, candidates, beam_width=25)
    result = evaluate_prediction(predicted, puzzle["groups"])
    solved_s3.append(result["solved"])
    correct_s3.append(result["n_correct"])
    top1_s3.append(result["top1_correct"])

print("Solvers 1-3 done")

Solvers 1-3 done


In [9]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/benjaminburda/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/benjaminburda/nltk_data...


True

In [ ]:
# ── Solver 4: Beam + penalty + lexical ───────────────────────────────
solved_s4, correct_s4, top1_s4 = [], [], []
for item in test_precomputed:
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    tau = item["tau"]
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    candidates = score_all_candidates(candidates, tau, ALPHA, BETA, GAMMA)
    candidates = add_lexical_scores(candidates)
    for c in candidates:
        c["score"] = c["score"] + ETA * c["lexical"]
    candidates.sort(key=lambda x: x["score"], reverse=True)
    predicted = beam_solve(words, embeddings, candidates, beam_width=25)
    result = evaluate_prediction(predicted, puzzle["groups"])
    solved_s4.append(result["solved"])
    correct_s4.append(result["n_correct"])
    top1_s4.append(result["top1_correct"])

print("Solver 4 done")

Solver 4 done


IndexError: list index out of range

In [13]:
# ── Solver 5 fixed: recompute embeddings after each correct guess ─────
solved_s5, correct_s5, top1_s5 = [], [], []
for item in test_precomputed:
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    tau = item["tau"]
    words = puzzle["words"]
    true_groups = puzzle["groups"]
    candidates = score_all_candidates(candidates, tau, ALPHA, BETA, GAMMA)
    candidates = add_lexical_scores(candidates)
    for c in candidates:
        c["score"] = c["score"] + ETA * c["lexical"]
    candidates.sort(key=lambda x: x["score"], reverse=True)

    remaining = set(words)
    solved_groups = []
    max_guesses = 7

    for _ in range(max_guesses):
        if len(solved_groups) == 4:
            break
        if len(remaining) < 4:
            break
        if len(remaining) == 4:
            solved_groups.append(list(remaining))
            break

        # recompute embeddings for remaining words
        remaining_words = list(remaining)
        remaining_embs = model.encode(remaining_words, show_progress_bar=False)

        active = [c for c in candidates if c["word_set"].issubset(remaining)]
        if not active:
            break

        try:
            predicted = beam_solve(remaining_words, remaining_embs, active, beam_width=25)
            guess = predicted[0]
        except (IndexError, Exception):
            break

        feedback = simulate_feedback(guess, true_groups)
        if feedback == "correct":
            solved_groups.append(guess)
            remaining -= set(guess)
        candidates = apply_feedback(candidates, guess, feedback, remaining)

    result = evaluate_prediction(solved_groups, true_groups)
    solved_s5.append(result["solved"])
    correct_s5.append(result["n_correct"])
    top1_s5.append(result["top1_correct"])

print("Solver 5 fixed done")
print(f"Solve rate:          {np.mean(solved_s5):.3f}")
print(f"Mean correct groups: {np.mean(correct_s5):.3f}")
print(f"Top-1 accuracy:      {np.mean(top1_s5):.3f}")

Solver 5 fixed done
Solve rate:          0.000
Mean correct groups: 0.739
Top-1 accuracy:      0.739


In [14]:
# ── Diagnostic: what's happening in S5 ──────────────────────────────
for item in test_precomputed[:5]:  # just first 5 puzzles
    puzzle = item["puzzle"]
    candidates = copy.deepcopy(item["candidates"])
    tau = item["tau"]
    words = puzzle["words"]
    true_groups = puzzle["groups"]
    candidates = score_all_candidates(candidates, tau, ALPHA, BETA, GAMMA)
    candidates = add_lexical_scores(candidates)
    for c in candidates:
        c["score"] = c["score"] + ETA * c["lexical"]
    candidates.sort(key=lambda x: x["score"], reverse=True)

    remaining = set(words)
    solved_groups = []
    print(f"\nPuzzle {puzzle['puzzle_id']} (difficulty {puzzle['difficulty']})")
    print(f"True groups: {[g['members'] for g in true_groups]}")

    for guess_num in range(7):
        if len(solved_groups) == 4 or len(remaining) < 4:
            break
        if len(remaining) == 4:
            solved_groups.append(list(remaining))
            break

        remaining_words = list(remaining)
        remaining_embs = model.encode(remaining_words, show_progress_bar=False)
        active = [c for c in candidates if c["word_set"].issubset(remaining)]
        if not active:
            break

        try:
            predicted = beam_solve(remaining_words, remaining_embs, active, beam_width=25)
            guess = predicted[0]
        except:
            break

        feedback = simulate_feedback(guess, true_groups)
        print(f"  Guess {guess_num+1}: {guess} → {feedback}")
        if feedback == "correct":
            solved_groups.append(guess)
            remaining -= set(guess)
        candidates = apply_feedback(candidates, guess, feedback, remaining)

    print(f"  Solved {len(solved_groups)}/4 groups")


Puzzle 313 (difficulty 2.6)
True groups: [['ADHERE', 'GLUE', 'PASTE', 'STICK'], ['COPY', 'TEXT', 'WORDS', 'WRITING'], ['CARAT', 'CLARITY', 'COLOR', 'CUT'], ['LIST', 'OK', 'PLUS', 'ROD']]
  Guess 1: ['ADHERE', 'GLUE', 'PASTE', 'STICK'] → correct
  Solved 1/4 groups

Puzzle 314 (difficulty 4.0)
True groups: [['BUNK', 'CROCK', 'HOGWASH', 'HORSEFEATHERS'], ['BATON', 'HAMMER', 'HURDLE', 'POLE'], ['GOATEE', 'HORNS', 'PITCHFORK', 'TAIL'], ['BEND', 'BOWLINE', 'HITCH', 'SHEEPSHANK']]
  Guess 1: ['BATON', 'HAMMER', 'POLE', 'PITCHFORK'] → one_away
  Guess 2: ['BATON', 'POLE', 'PITCHFORK', 'TAIL'] → incorrect
  Guess 3: ['BATON', 'HAMMER', 'HORNS', 'PITCHFORK'] → incorrect
  Guess 4: ['BATON', 'HAMMER', 'GOATEE', 'PITCHFORK'] → incorrect
  Guess 5: ['BATON', 'HAMMER', 'PITCHFORK', 'SHEEPSHANK'] → incorrect
  Guess 6: ['BATON', 'HAMMER', 'POLE', 'TAIL'] → one_away
  Guess 7: ['HAMMER', 'POLE', 'PITCHFORK', 'TAIL'] → incorrect
  Solved 0/4 groups

Puzzle 315 (difficulty 3.8)
True groups: [['EXAMPLE

In [15]:
# ── Check what happens after correct guess ────────────────────────────
item = test_precomputed[0]  # puzzle 313
puzzle = item["puzzle"]
candidates = copy.deepcopy(item["candidates"])
tau = item["tau"]
words = puzzle["words"]
true_groups = puzzle["groups"]

candidates = score_all_candidates(candidates, tau, ALPHA, BETA, GAMMA)
candidates = add_lexical_scores(candidates)
for c in candidates:
    c["score"] = c["score"] + ETA * c["lexical"]
candidates.sort(key=lambda x: x["score"], reverse=True)

remaining = set(words)
print(f"Starting words: {len(remaining)}")

# simulate first correct guess
guess = ['ADHERE', 'GLUE', 'PASTE', 'STICK']
remaining -= set(guess)
candidates = apply_feedback(candidates, guess, "correct", remaining)

print(f"Remaining words after correct guess: {len(remaining)}")
print(f"Remaining words: {remaining}")
print(f"Active candidates after feedback: {len([c for c in candidates if c['word_set'].issubset(remaining)])}")

Starting words: 16
Remaining words after correct guess: 12
Remaining words: {'CLARITY', 'TEXT', 'PLUS', 'COPY', 'WORDS', 'COLOR', 'CUT', 'OK', 'CARAT', 'ROD', 'WRITING', 'LIST'}
Active candidates after feedback: 495


In [16]:
# ── Check embedding mismatch ──────────────────────────────────────────
remaining_words = list(remaining)
print(f"Remaining words: {len(remaining_words)}")

remaining_embs = model.encode(remaining_words, show_progress_bar=False)
print(f"Remaining embeddings shape: {remaining_embs.shape}")

active = [c for c in candidates if c["word_set"].issubset(remaining)]
print(f"Active candidates: {len(active)}")

try:
    predicted = beam_solve(remaining_words, remaining_embs, active, beam_width=25)
    print(f"Beam solve result: {predicted}")
except Exception as e:
    print(f"Error: {e}")

Remaining words: 12
Remaining embeddings shape: (12, 768)
Active candidates: 495
Error: list index out of range


In [12]:
# ── Full ablation results table ──────────────────────────────────────
results_summary = pd.DataFrame([
    {
        "Solver": "S1: Greedy",
        "Solve Rate": np.mean(solved_s1),
        "Mean Correct Groups": np.mean(correct_s1),
        "Top-1 Accuracy": np.mean(top1_s1)
    },
    {
        "Solver": "S2: Beam",
        "Solve Rate": np.mean(solved_s2),
        "Mean Correct Groups": np.mean(correct_s2),
        "Top-1 Accuracy": np.mean(top1_s2)
    },
    {
        "Solver": "S3: Beam + Penalty",
        "Solve Rate": np.mean(solved_s3),
        "Mean Correct Groups": np.mean(correct_s3),
        "Top-1 Accuracy": np.mean(top1_s3)
    },
    {
        "Solver": "S4: Beam + Penalty + Lexical",
        "Solve Rate": np.mean(solved_s4),
        "Mean Correct Groups": np.mean(correct_s4),
        "Top-1 Accuracy": np.mean(top1_s4)
    },
    {
        "Solver": "S5: Full Pipeline + Feedback",
        "Solve Rate": np.mean(solved_s5),
        "Mean Correct Groups": np.mean(correct_s5),
        "Top-1 Accuracy": np.mean(top1_s5)
    }
])

print(results_summary.round(3).to_string(index=False))

                      Solver  Solve Rate  Mean Correct Groups  Top-1 Accuracy
                  S1: Greedy       0.000                0.457           0.217
                    S2: Beam       0.065                0.717           0.261
          S3: Beam + Penalty       0.065                0.717           0.261
S4: Beam + Penalty + Lexical       0.043                0.739           0.348
S5: Full Pipeline + Feedback       0.000                0.739           0.739


In [7]:
# ── Solver 1: Greedy baseline ─────────────────────────────────────────
print("Running Solver 1: Greedy baseline...")
results_greedy = run_evaluation(
    test_puzzles,
    solver="greedy",
    use_lexical=False,
    use_feedback=False
)

print(f"Solve rate:            {results_greedy['solve_rate']:.3f}")
print(f"Mean correct groups:   {results_greedy['mean_correct_groups']:.3f}")
print(f"Top-1 accuracy:        {results_greedy['top1_accuracy']:.3f}")
print(f"N puzzles evaluated:   {results_greedy['n_puzzles']}")

Running Solver 1: Greedy baseline...
Solve rate:            0.000
Mean correct groups:   0.500
Top-1 accuracy:        0.130
N puzzles evaluated:   46


In [8]:
# ── Solver 2: Beam search ─────────────────────────────────────────────
print("Running Solver 2: Beam search...")
results_beam = run_evaluation(
    test_puzzles,
    solver="beam",
    beam_width=25,
    use_lexical=False,
    use_feedback=False
)

print(f"Solve rate:            {results_beam['solve_rate']:.3f}")
print(f"Mean correct groups:   {results_beam['mean_correct_groups']:.3f}")
print(f"Top-1 accuracy:        {results_beam['top1_accuracy']:.3f}")
print(f"N puzzles evaluated:   {results_beam['n_puzzles']}")

Running Solver 2: Beam search...
Solve rate:            0.043
Mean correct groups:   0.674
Top-1 accuracy:        0.239
N puzzles evaluated:   46


In [9]:
# ── Solver 3: Beam + false group penalty ─────────────────────────────
print("Running Solver 3: Beam + false group penalty...")
results_beam_penalty = run_evaluation(
    test_puzzles,
    solver="beam",
    beam_width=25,
    alpha=1.0,
    beta=0.5,
    gamma=0.1,
    use_lexical=False,
    use_feedback=False
)

print(f"Solve rate:            {results_beam_penalty['solve_rate']:.3f}")
print(f"Mean correct groups:   {results_beam_penalty['mean_correct_groups']:.3f}")
print(f"Top-1 accuracy:        {results_beam_penalty['top1_accuracy']:.3f}")
print(f"N puzzles evaluated:   {results_beam_penalty['n_puzzles']}")

Running Solver 3: Beam + false group penalty...
Solve rate:            0.043
Mean correct groups:   0.674
Top-1 accuracy:        0.239
N puzzles evaluated:   46


In [11]:
# ── Precompute candidates for all training puzzles ────────────────────
from solver.candidates import generate_candidates
from solver.scoring import score_all_candidates
from solver.search import get_tau_estimate

print("Precomputing candidates for training puzzles...")
train_candidates = []
for puzzle in train_puzzles:
    words = puzzle["words"]
    embeddings = np.array(puzzle["embeddings"])
    candidates = generate_candidates(words, embeddings)
    tau = get_tau_estimate(candidates)
    train_candidates.append({
        "puzzle": puzzle,
        "candidates": candidates,
        "tau": tau
    })
print("Done")

Precomputing candidates for training puzzles...
Done


In [12]:
# ── Grid search on precomputed candidates ────────────────────────────
from solver.search import beam_solve
from solver.evaluate import evaluate_prediction
import copy

betas  = [0.0, 0.25, 0.50, 0.75, 1.0]
gammas = [0.0, 0.05, 0.10, 0.20, 0.30]

best_score = -1
best_weights = None
tuning_results = []

total = len(betas) * len(gammas)
print(f"Testing {total} weight combinations...")

for b, g in itertools.product(betas, gammas):
    solved = []
    correct_groups = []
    top1s = []
    
    for item in train_candidates:
        puzzle = item["puzzle"]
        tau = item["tau"]
        # deep copy so we don't modify original
        candidates = copy.deepcopy(item["candidates"])
        candidates = score_all_candidates(candidates, tau, 
                                         alpha=1.0, beta=b, gamma=g)
        words = puzzle["words"]
        embeddings = np.array(puzzle["embeddings"])
        predicted = beam_solve(words, embeddings, candidates, beam_width=25)
        result = evaluate_prediction(predicted, puzzle["groups"])
        solved.append(result["solved"])
        correct_groups.append(result["n_correct"])
        top1s.append(result["top1_correct"])
    
    solve_rate = np.mean(solved)
    tuning_results.append({
        "beta": b, "gamma": g,
        "solve_rate": solve_rate,
        "mean_correct": np.mean(correct_groups),
        "top1": np.mean(top1s)
    })
    if solve_rate > best_score:
        best_score = solve_rate
        best_weights = (1.0, b, g)
    print(f"  beta={b:.2f} gamma={g:.2f} → solve_rate={solve_rate:.3f}")

tuning_df = pd.DataFrame(tuning_results).sort_values("solve_rate", ascending=False)
print(f"\nBest weights: alpha=1.0, beta={best_weights[1]}, gamma={best_weights[2]}")
print(f"Best training solve rate: {best_score:.3f}")
print("\nTop 10 combinations:")
print(tuning_df.head(10).to_string(index=False))

Testing 25 weight combinations...
  beta=0.00 gamma=0.00 → solve_rate=0.056
  beta=0.00 gamma=0.05 → solve_rate=0.061
  beta=0.00 gamma=0.10 → solve_rate=0.056
  beta=0.00 gamma=0.20 → solve_rate=0.050
  beta=0.00 gamma=0.30 → solve_rate=0.050
  beta=0.25 gamma=0.00 → solve_rate=0.039
  beta=0.25 gamma=0.05 → solve_rate=0.056
  beta=0.25 gamma=0.10 → solve_rate=0.050
  beta=0.25 gamma=0.20 → solve_rate=0.050
  beta=0.25 gamma=0.30 → solve_rate=0.050
  beta=0.50 gamma=0.00 → solve_rate=0.039
  beta=0.50 gamma=0.05 → solve_rate=0.039
  beta=0.50 gamma=0.10 → solve_rate=0.039
  beta=0.50 gamma=0.20 → solve_rate=0.039
  beta=0.50 gamma=0.30 → solve_rate=0.039
  beta=0.75 gamma=0.00 → solve_rate=0.044
  beta=0.75 gamma=0.05 → solve_rate=0.039
  beta=0.75 gamma=0.10 → solve_rate=0.039
  beta=0.75 gamma=0.20 → solve_rate=0.039
  beta=0.75 gamma=0.30 → solve_rate=0.039
  beta=1.00 gamma=0.00 → solve_rate=0.044
  beta=1.00 gamma=0.05 → solve_rate=0.039
  beta=1.00 gamma=0.10 → solve_rate=0.039
